## **SNAP Jupyter demo notebook**
**Sentinel-1 ETAD correction and ETAD-enhanced InSAR**

In summary, this workflow contains:

- Background on the **ETAD** (Extended Timing Annotation Dataset) auxiliary product and the per-pixel corrections it carries
- **Part 1**: apply ETAD to a single Sentinel-1 IW SLC burst, output the phase correction bands, and inspect what the correction looks like
- **Part 2**: full ETAD-enhanced IW InSAR pair (classical chain with ETAD inserted after `TOPSAR-Split`)
- **Part 3** (optional): run the same InSAR pair **without** ETAD and compare the two phase results side-by-side

Complexity: advanced

##### ***About the test data:***

Each Sentinel-1 SLC needs its **matching ETAD product** — same acquisition time, same orbit, same datatake. ETAD product filenames look like `S1*_*_ETA__AX*` and are around 200 MB each (much smaller than the SLC).

Download an interferometric pair from the [Copernicus Browser](https://dataspace.copernicus.eu/browser/), and for each SLC also download its ETAD counterpart — filter the catalogue by product type `S1_ETA`. You should end up with **four** products:

- **Reference SLC** + reference ETAD (same acquisition timestamp)
- **Secondary SLC** + secondary ETAD

Place all four under the notebook's `data/` folder and update the path variables below.

ETAD production began in 2022; older Sentinel-1 acquisitions may not have an ETAD counterpart available. Pick a 2022+ pair to be safe.

##### ***Some information on the Python environment:***

In [ ]:
import os
import sys
print("Python version: " + sys.version)

import sysconfig
print("Location of esa_snappy package: " + sysconfig.get_paths()['purelib'] + os.sep + "esa_snappy")

##### ***Import Python packages...***

In [ ]:
import esa_snappy
from esa_snappy import ProductIO

import snapista
from snapista import Graph
from snapista import Operator

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

##### ***Convenience plot functions:***

In [ ]:
def _read_band(product, name):
    band = product.getBand(name)
    if band is None:
        raise KeyError(f"Band {name!r} not found. Available: {[b.getName() for b in product.getBands()]}")
    w, h = band.getRasterWidth(), band.getRasterHeight()
    data = np.zeros(w * h, np.float32)
    band.readPixels(0, 0, w, h, data)
    data.shape = h, w
    return data

def plot_band(product, name, title=None, cmap='viridis', vmin=None, vmax=None):
    data = _read_band(product, name)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title or name)
    fig.colorbar(im, ax=ax)
    plt.show()

def plot_interferogram(product, phase_band_name, coh_band_name=None):
    phi = _read_band(product, phase_band_name)
    if coh_band_name is None:
        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        ax.set_title('Wrapped interferogram phase [rad]')
        fig.colorbar(im, ax=ax)
    else:
        coh = _read_band(product, coh_band_name)
        fig, axs = plt.subplots(1, 2, figsize=(13, 5))
        im0 = axs[0].imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        axs[0].set_title('Wrapped interferogram phase [rad]')
        fig.colorbar(im0, ax=axs[0])
        im1 = axs[1].imshow(coh, cmap='viridis', vmin=0, vmax=1)
        axs[1].set_title('Coherence [0–1]')
        fig.colorbar(im1, ax=axs[1])
    plt.show()

def plot_phase_side_by_side(prod_a, phase_a, label_a, prod_b, phase_b, label_b):
    """Plot two wrapped-phase bands from two products on a shared colour scale."""
    phi_a = _read_band(prod_a, phase_a)
    phi_b = _read_band(prod_b, phase_b)
    fig, axs = plt.subplots(1, 2, figsize=(13, 5))
    for ax, phi, label in zip(axs, [phi_a, phi_b], [label_a, label_b]):
        im = ax.imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        ax.set_title(label + ' — wrapped phase [rad]')
        fig.colorbar(im, ax=ax)
    plt.show()

def find_band(product, *patterns):
    names = [b.getName() for b in product.getBands()]
    for pat in patterns:
        for n in names:
            if pat.lower() in n.lower():
                return n
    raise KeyError(f"No band matching {patterns!r} found. Available: {names}")

---

### ***Background: what is ETAD?***

ETAD — the **Extended Timing Annotation Dataset** — is an ESA auxiliary product that carries per-pixel range and azimuth time corrections (and the derived phase corrections) for a single Sentinel-1 SLC acquisition. It bundles four classes of effects that the SLC's own annotation cannot account for:

| Class | Component | Effect |
|:------|:----------|:-------|
| Geodetic | Solid Earth tide, ocean tide loading, bistatic correction | centimetres of geolocation; non-negligible phase for short baselines |
| Geometric | Doppler centroid shift, FM-rate mismatch | sub-pixel azimuth shifts, especially near burst edges in IW |
| Tropospheric | Wet + dry zenith path delays | up to several radians of differential phase, slowly varying spatially |
| Ionospheric | Total electron content (TEC) | strongly varying with solar activity; the dominant InSAR error at high latitudes |

Applied to an SLC, an ETAD correction sharpens absolute geolocation to a few centimetres and — most relevant here — removes deterministic phase contributions that would otherwise show up as low-frequency "ramps" or "bubbles" in your differential interferogram.

In SNAP the operator is `S1-ETAD-Correction`. It takes the SLC as the source product and the matching `.SAFE` ETAD product via the `etadFile` parameter.

---

### ***Background: where ETAD fits in the InSAR chain***

The classical Sentinel-1 IW InSAR chain is:

```text
Read → Apply-Orbit-File → TOPSAR-Split → Back-Geocoding → ESD → Interferogram
     → TOPSAR-Deburst → TopoPhaseRemoval → Goldstein → Terrain-Correction
```

ETAD slots in **between `TOPSAR-Split` and `Back-Geocoding`**, applied independently to reference and secondary. The full ETAD-enhanced chain is:

```text
Read → Apply-Orbit-File → TOPSAR-Split → S1-ETAD-Correction → Back-Geocoding → ...
```

Why there?

- **After `TOPSAR-Split`** so ETAD only operates on the subswath/bursts you actually want to process (the ETAD `.SAFE` carries per-burst data — splitting first keeps it efficient).
- **Before `Back-Geocoding`** so the geometric corrections inside ETAD improve the back-geocoding lookup table, *and* so the phase corrections are baked into the SLC before interferogram formation.

Two key parameters of `S1-ETAD-Correction`:

| Parameter | Recommended | Effect |
|:----------|:------------|:-------|
| `etadFile` | path to matching `S1*_ETA*.SAFE/manifest.safe` | which ETAD to apply |
| `resamplingImage` | `true` for InSAR | shift pixels per the ETAD timing corrections |
| `outputPhaseCorrections` | `false` for production, `true` for inspection | add per-pixel phase correction bands to the output |

---

### ***Configure input paths***

Two SLCs, two ETAD products. Update to your downloaded files.

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')
graphs_dir = os.path.join(os.getcwd(), 'graphs')
results_dir = os.path.join(os.getcwd(), 'results')
os.makedirs(graphs_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# Reference
reference_slc = os.path.join(data_dir, 'S1A_IW_SLC__1SDV_<reference>.SAFE', 'manifest.safe')
reference_etad = os.path.join(data_dir, 'S1A_IW_ETA__AXDV_<reference>.SAFE', 'manifest.safe')

# Secondary
secondary_slc = os.path.join(data_dir, 'S1A_IW_SLC__1SDV_<secondary>.SAFE', 'manifest.safe')
secondary_etad = os.path.join(data_dir, 'S1A_IW_ETA__AXDV_<secondary>.SAFE', 'manifest.safe')

# Subswath + burst range to process (keep small for demo runtime)
subswath = 'IW1'
polarisation = 'VV'
first_burst = '1'
last_burst = '3'

---
## ***Part 1 — Inspect what ETAD adds to a single SLC***
---

We process just the reference SLC: `Read → Apply-Orbit-File → TOPSAR-Split → S1-ETAD-Correction` with `outputPhaseCorrections=true` so the per-pixel phase correction bands are written to the output product. `resamplingImage=false` here — we don't actually shift the pixels, we just want to see the correction.

The output product will contain:

- the original (unshifted) `i_*` / `q_*` complex bands,
- phase-correction bands such as `etadPhaseCorrection*` (one per correction type or a combined total, depending on SNAP version).

In [ ]:
g_inspect = Graph()
g_inspect.add_node(operator=Operator('Read', file=reference_slc), node_id='Read')
g_inspect.add_node(operator=Operator('Apply-Orbit-File',
                                     orbitType='Sentinel Precise (Auto Download)',
                                     continueOnFail='true'),
                   node_id='Orbit', source='Read')
g_inspect.add_node(operator=Operator('TOPSAR-Split',
                                     subswath=subswath,
                                     selectedPolarisations=polarisation,
                                     firstBurstIndex=first_burst,
                                     lastBurstIndex=last_burst),
                   node_id='Split', source='Orbit')
g_inspect.add_node(operator=Operator('S1-ETAD-Correction',
                                     etadFile=reference_etad,
                                     outputPhaseCorrections='true',
                                     resamplingImage='false'),
                   node_id='ETAD', source='Split')

inspect_out = os.path.join(results_dir, 'snap_nb_etad_inspect.dim')
g_inspect.add_node(operator=Operator('Write', file=inspect_out, formatName='BEAM-DIMAP'),
                   node_id='Write', source='ETAD')

g_inspect.view()
g_inspect.save_graph(os.path.join(graphs_dir, 'snap_nb_etad_inspect.xml'))
g_inspect.run()

##### ***Inspect the phase correction bands:***

In [ ]:
p_inspect = ProductIO.readProduct(inspect_out)
print('Bands:', [b.getName() for b in p_inspect.getBands()])

# Find any band whose name suggests it carries an ETAD phase correction.
phase_corr_names = [b.getName() for b in p_inspect.getBands()
                    if 'etad' in b.getName().lower() or 'corrPhase' in b.getName().lower()
                    or 'phaseCorrection' in b.getName().lower()]
print('Phase correction bands:', phase_corr_names)

for name in phase_corr_names[:4]:  # cap at 4 for readability
    plot_band(p_inspect, name, title=f'ETAD correction: {name}', cmap='RdBu_r')

p_inspect.dispose()

---
## ***Part 2 — ETAD-enhanced IW InSAR pair***
---

The full chain now. ETAD is applied independently to reference and secondary with `resamplingImage=true` so the corrections are baked into the SLC pixels before back-geocoding and interferogram formation.

Pipeline:

```text
                Reference SLC               Secondary SLC
                     │                          │
             Apply-Orbit-File          Apply-Orbit-File
                     │                          │
             TOPSAR-Split              TOPSAR-Split
                     │                          │
        S1-ETAD-Correction         S1-ETAD-Correction
                     │                          │
                     └───── Back-Geocoding ────┘
                              │
                Enhanced-Spectral-Diversity
                              │
                       Interferogram
                              │
                      TOPSAR-Deburst
                              │
                  GoldsteinPhaseFiltering
                              │
                           Write
```

In [ ]:
def build_iw_branch(g, slc_path, etad_path, tag):
    """Add Read → Apply-Orbit-File → TOPSAR-Split → S1-ETAD-Correction nodes to `g` and return the terminal node_id."""
    g.add_node(operator=Operator('Read', file=slc_path),
               node_id=f'Read_{tag}')
    g.add_node(operator=Operator('Apply-Orbit-File',
                                 orbitType='Sentinel Precise (Auto Download)',
                                 continueOnFail='true'),
               node_id=f'Orbit_{tag}', source=f'Read_{tag}')
    g.add_node(operator=Operator('TOPSAR-Split',
                                 subswath=subswath,
                                 selectedPolarisations=polarisation,
                                 firstBurstIndex=first_burst,
                                 lastBurstIndex=last_burst),
               node_id=f'Split_{tag}', source=f'Orbit_{tag}')
    g.add_node(operator=Operator('S1-ETAD-Correction',
                                 etadFile=etad_path,
                                 outputPhaseCorrections='false',
                                 resamplingImage='true'),
               node_id=f'ETAD_{tag}', source=f'Split_{tag}')
    return f'ETAD_{tag}'

g_insar = Graph()
ref_node = build_iw_branch(g_insar, reference_slc, reference_etad, 'ref')
sec_node = build_iw_branch(g_insar, secondary_slc, secondary_etad, 'sec')

# Back-Geocoding takes both ETAD-corrected branches as inputs
g_insar.add_node(operator=Operator('Back-Geocoding',
                                   demName='Copernicus 30m Global DEM',
                                   resamplingType='BISINC_5_POINT_INTERPOLATION'),
                 node_id='BackGeo', source=[ref_node, sec_node])

# Enhanced-Spectral-Diversity for azimuth refinement (TOPS only)
g_insar.add_node(operator=Operator('Enhanced-Spectral-Diversity'),
                 node_id='ESD', source='BackGeo')

# Interferogram with flat-earth and (optional) topo-phase removal
g_insar.add_node(operator=Operator('Interferogram',
                                   subtractFlatEarthPhase='true',
                                   subtractTopographicPhase='true',
                                   demName='Copernicus 30m Global DEM',
                                   includeCoherence='true',
                                   cohWinAz='10',
                                   cohWinRg='10'),
                 node_id='Ifg', source='ESD')

g_insar.add_node(operator=Operator('TOPSAR-Deburst'),
                 node_id='Deburst', source='Ifg')
g_insar.add_node(operator=Operator('GoldsteinPhaseFiltering', alpha='1.0'),
                 node_id='Goldstein', source='Deburst')

etad_insar_out = os.path.join(results_dir, 'snap_nb_etad_insar.dim')
g_insar.add_node(operator=Operator('Write', file=etad_insar_out, formatName='BEAM-DIMAP'),
                 node_id='Write', source='Goldstein')

g_insar.save_graph(os.path.join(graphs_dir, 'snap_nb_etad_insar.xml'))
g_insar.run()

In [ ]:
p_etad_ifg = ProductIO.readProduct(etad_insar_out)
phase = find_band(p_etad_ifg, 'Phase_ifg', 'phase_')
coh = find_band(p_etad_ifg, 'coh_')
plot_interferogram(p_etad_ifg, phase, coh)
# Don't dispose yet — we may want to compare against the non-ETAD version below

---
## ***Part 3 (optional) — compare with and without ETAD***
---

To see what ETAD actually buys us, run **the same chain but with the `S1-ETAD-Correction` nodes removed** and compare the two interferograms.

The visible difference is usually subtle: low-frequency phase ramps (tropospheric / ionospheric) that look like "unrelated noise" in the no-ETAD version disappear in the ETAD-corrected version. The improvement is most visible at high latitudes (ionosphere) or when one acquisition has weather contamination (troposphere).

This roughly doubles the notebook runtime — skip if you're short on time.

In [ ]:
def build_iw_branch_no_etad(g, slc_path, tag):
    g.add_node(operator=Operator('Read', file=slc_path),
               node_id=f'Read_{tag}')
    g.add_node(operator=Operator('Apply-Orbit-File',
                                 orbitType='Sentinel Precise (Auto Download)',
                                 continueOnFail='true'),
               node_id=f'Orbit_{tag}', source=f'Read_{tag}')
    g.add_node(operator=Operator('TOPSAR-Split',
                                 subswath=subswath,
                                 selectedPolarisations=polarisation,
                                 firstBurstIndex=first_burst,
                                 lastBurstIndex=last_burst),
               node_id=f'Split_{tag}', source=f'Orbit_{tag}')
    return f'Split_{tag}'

g_noetad = Graph()
ref_node = build_iw_branch_no_etad(g_noetad, reference_slc, 'ref')
sec_node = build_iw_branch_no_etad(g_noetad, secondary_slc, 'sec')

g_noetad.add_node(operator=Operator('Back-Geocoding',
                                    demName='Copernicus 30m Global DEM',
                                    resamplingType='BISINC_5_POINT_INTERPOLATION'),
                  node_id='BackGeo', source=[ref_node, sec_node])
g_noetad.add_node(operator=Operator('Enhanced-Spectral-Diversity'),
                  node_id='ESD', source='BackGeo')
g_noetad.add_node(operator=Operator('Interferogram',
                                    subtractFlatEarthPhase='true',
                                    subtractTopographicPhase='true',
                                    demName='Copernicus 30m Global DEM',
                                    includeCoherence='true',
                                    cohWinAz='10',
                                    cohWinRg='10'),
                  node_id='Ifg', source='ESD')
g_noetad.add_node(operator=Operator('TOPSAR-Deburst'),
                  node_id='Deburst', source='Ifg')
g_noetad.add_node(operator=Operator('GoldsteinPhaseFiltering', alpha='1.0'),
                  node_id='Goldstein', source='Deburst')

noetad_out = os.path.join(results_dir, 'snap_nb_etad_insar_NO_ETAD.dim')
g_noetad.add_node(operator=Operator('Write', file=noetad_out, formatName='BEAM-DIMAP'),
                  node_id='Write', source='Goldstein')

g_noetad.save_graph(os.path.join(graphs_dir, 'snap_nb_etad_insar_NO_ETAD.xml'))
g_noetad.run()

In [ ]:
p_noetad_ifg = ProductIO.readProduct(noetad_out)

phase_with = find_band(p_etad_ifg, 'Phase_ifg', 'phase_')
phase_without = find_band(p_noetad_ifg, 'Phase_ifg', 'phase_')

plot_phase_side_by_side(p_noetad_ifg, phase_without, 'WITHOUT ETAD',
                        p_etad_ifg, phase_with, 'WITH ETAD')

p_etad_ifg.dispose()
p_noetad_ifg.dispose()

---

### ***Summary***

What have we learnt in this notebook?

- An **ETAD** product is the per-pixel correction lookup table for a single Sentinel-1 SLC. It bundles geodetic, geometric, tropospheric and ionospheric effects.
- The SNAP operator `S1-ETAD-Correction` takes the SLC as source and the matching ETAD `.SAFE` via `etadFile`. Two key parameters:
  - `outputPhaseCorrections=true` to emit the correction bands for inspection
  - `resamplingImage=true` to actually shift the SLC pixels per the ETAD timing corrections (recommended for InSAR)
- In the InSAR chain, ETAD slots in **between `TOPSAR-Split` and `Back-Geocoding`**, applied independently to reference and secondary. The rest of the classical IW InSAR chain is unchanged.
- Comparing the same interferogram with and without ETAD highlights low-frequency atmospheric phase ramps that are removed by ETAD correction — particularly important for short-baseline displacement work and at high latitudes.